# Lab: Open Text-to-Speech, Voice Cloning and Speaker Verification

In this lab you will use **open-weight, state-of-the-art speech models** that run entirely inside this Colab session. No cloud API keys or accounts are needed.

| Part | What you do | Model(s) | Time in class |
|---|---|---|---|
| 0 | Set up the runtime | | ~3 min |
| 1 | Record **your own voice** | browser microphone | ~5 min |
| 2 | Classic multi-voice TTS and grapheme-to-phoneme | **Kokoro-82M** (Apache 2.0) | ~10 min |
| 3 | Zero-shot **voice cloning** from a few seconds of your speech | **Qwen3-TTS-12Hz-0.6B-Base** (Apache 2.0) | ~15 min |
| 4 | Measure intelligibility with ASR (round trip) | **Whisper large-v3-turbo** | ~5 min |
| 5 | **Speaker verification and identification**: can your clone fool a speaker-verification system? | **ECAPA-TDNN** (SpeechBrain) | ~15 min |


**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better). Run cells top to bottom.



---
## Part 0: Setup

This installs everything in one go (2–3 minutes). `qwen-tts` pins a specific `transformers` version; if Colab asks you to **restart the session** after this cell, do so and then continue from the *next* cell.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y espeak-ng sox ffmpeg > /dev/null
!pip install -q --ignore-requires-python --no-deps "kokoro>=0.9.4" "misaki>=0.9.4"
!pip install -q loguru addict regex espeakng-loader num2words phonemizer-fork spacy
!pip install -q -U qwen-tts speechbrain jiwer

In [ ]:
import os, io, json, base64, subprocess, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch, torch.nn.functional as F
import librosa, librosa.display, soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    major, _ = torch.cuda.get_device_capability()
    gpu = torch.cuda.get_device_name()
    # T4 (compute capability 7.5) has no native bfloat16 -> use float32 for safety.
    DTYPE = torch.bfloat16 if major >= 8 else torch.float32
    print(f"GPU: {gpu}  | dtype for Qwen3-TTS: {DTYPE}")
else:
    DTYPE = torch.float32
    print("⚠️ No GPU found. Parts 2 and 5 will work on CPU; Parts 3, 4 and 6 will be very slow.")

SR = 24000                     # Kokoro and Qwen3-TTS both work at 24 kHz
os.makedirs("/content/my_voice", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

# A small "audio bank": every clip we make or record goes here, so later parts can analyse it.
# name -> dict(y=np.ndarray, sr=int, text=str, kind=str)
BANK = {}

def add(name, y, sr, text, kind, show=True):
    y = np.asarray(y, dtype=np.float32).squeeze()
    BANK[name] = dict(y=y, sr=sr, text=text, kind=kind)
    sf.write(f"/content/outputs/{name}.wav", y, sr)
    if show:
        print(f"[{kind}] {name}: {len(y)/sr:.1f}s")
        display(Audio(y, rate=sr))
    return y

def to16k(y, sr):
    return librosa.resample(y, orig_sr=sr, target_sr=16000) if sr != 16000 else y

---
## Part 1: Record your own voice

As in the speech recognition lab, you will record yourself in the browser. Each prompt shows a **⏺ Start** button; click it, read the sentence aloud, then click **⏹ Stop** (recording stops automatically after 15 s).

Tips: quiet room, headset microphone if you have one, normal speaking voice, about 20–30 cm from the mic. Leave a short pause before and after speaking (silence is trimmed automatically).

If the browser microphone doesn't work, use the **upload fallback** cell below to upload WAV/MP3/M4A files recorded on your phone instead.

In [ ]:
from google.colab import output

RECORD_JS = r'''
async function recordAudio(maxMs, promptText) {
  const div = document.createElement('div');
  div.style = 'font-family:sans-serif;padding:10px;border:1px solid #bbb;border-radius:8px;margin:6px 0;max-width:700px';
  div.innerHTML = '<div style="margin-bottom:8px"><b>Read aloud:</b> <span style="font-size:1.15em">' + promptText + '</span></div>';
  const btn = document.createElement('button');
  btn.textContent = '⏺ Start recording';
  btn.style = 'font-size:1em;padding:4px 12px';
  const status = document.createElement('span');
  status.style = 'margin-left:10px';
  div.appendChild(btn); div.appendChild(status);
  document.body.appendChild(div);
  await new Promise(r => btn.onclick = r);
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  btn.textContent = '⏹ Stop';
  status.textContent = '🔴 recording…';
  await new Promise(r => { btn.onclick = r; setTimeout(r, maxMs); });
  const stopped = new Promise(r => rec.onstop = r);
  rec.stop(); await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();
  const blob = new Blob(chunks, {type: rec.mimeType});
  return await new Promise(r => { const fr = new FileReader(); fr.onloadend = () => r(fr.result); fr.readAsDataURL(blob); });
}
'''

def convert_to_wav(src_path, dst_path, sr=SR, top_db=35):
    # Any audio format -> mono, 24 kHz WAV, leading/trailing silence trimmed, peak-normalised.
    tmp = dst_path + ".tmp.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src_path, "-ac", "1", "-ar", str(sr), tmp], check=True)
    y, _ = librosa.load(tmp, sr=sr)
    os.remove(tmp)
    y, _ = librosa.effects.trim(y, top_db=top_db)
    pad = np.zeros(int(0.15 * sr), dtype=np.float32)          # keep a little silence either side
    y = np.concatenate([pad, y, pad])
    y = 0.9 * y / (np.abs(y).max() + 1e-8)
    sf.write(dst_path, y, sr)
    return y

def record(prompt, dst_path, max_seconds=15):
    data_url = output.eval_js(RECORD_JS + f"recordAudio({int(max_seconds*1000)}, {json.dumps(prompt)})")
    raw = base64.b64decode(data_url.split(",", 1)[1])
    with open("/tmp/rec.webm", "wb") as f:
        f.write(raw)
    return convert_to_wav("/tmp/rec.webm", dst_path)

def record_set(prompts, folder, prefix):
    os.makedirs(folder, exist_ok=True)
    paths = []
    for i, p in enumerate(prompts):
        path = f"{folder}/{prefix}{i:03d}.wav"
        y = record(p, path)
        print(f"{i}: {len(y)/SR:.1f}s  {p}")
        display(Audio(y, rate=SR))
        paths.append(path)
    return paths

The first prompt is your **reference clip** for cloning (aim for 8–12 s). The other four are **held-out** sentences that we will use to test speaker verification and compare against the clones.

In [ ]:
MY_PROMPTS = [
    # 0: reference clip for cloning
    "When the sunlight strikes raindrops in the air, they act as a prism and form a rainbow. "
    "The rainbow is a division of white light into many beautiful colours.",
    # 1-4: held-out test sentences
    "Mary had a little lamb, its fleece was white as snow.",
    "I'm a person with a distinctive voice.",
    "The quick brown fox jumps over the lazy dog.",
    "Natural language technology turns text into speech and speech into text.",
]
my_paths = [f"/content/my_voice/me_{i:03d}.wav" for i in range(len(MY_PROMPTS))]

In [ ]:
my_paths = record_set(MY_PROMPTS, "/content/my_voice", "me_")

In [ ]:
# Add the recordings to the audio bank and look at the reference clip
for i, (p, t) in enumerate(zip(my_paths, MY_PROMPTS)):
    y, _ = librosa.load(p, sr=SR)
    add(f"me_{i}", y, SR, t, "real", show=False)

REF_PATH, REF_TEXT = my_paths[0], MY_PROMPTS[0]
y = BANK["me_0"]["y"]
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
librosa.display.waveshow(y, sr=SR, ax=ax[0]); ax[0].set_title("Your reference clip")
S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=256)), ref=np.max)
librosa.display.specshow(S, sr=SR, hop_length=256, x_axis="time", y_axis="log", ax=ax[1])
plt.tight_layout(); plt.show()
print({k: f"{len(v['y'])/SR:.1f}s" for k, v in BANK.items()})

---
## Part 2: Classic TTS with Kokoro-82M

[Kokoro](https://huggingface.co/hexgrad/Kokoro-82M) is a small (82 M parameter) open TTS model based on StyleTTS 2 with an iSTFTNet vocoder. Despite its size it competes with much larger systems, and it runs in real time even on CPU. It replaces the Google Cloud voice (`en-GB-Neural2-C`) we used in previous years.

The pipeline is the classic two-stage design:
1. **Text → phonemes** (grapheme-to-phoneme, G2P) using the `misaki` library (dictionary + espeak-ng fallback),
2. **Phonemes + voice style vector → waveform** (the neural acoustic model + vocoder).

Kokoro has no voice cloning: each voice is a fixed pre-computed *style vector*.

In [ ]:
from kokoro import KPipeline

kokoro = {
    "b": KPipeline(lang_code="b", repo_id="hexgrad/Kokoro-82M"),   # British English
    "a": KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M"),   # American English
}

def kokoro_tts(text, voice="bf_emma", speed=1.0, verbose=True):
    pipe = kokoro[voice[0]]            # first letter of the voice name = language/accent
    chunks, phonemes = [], []
    for gs, ps, audio in pipe(text, voice=voice, speed=speed):
        chunks.append(audio.cpu().numpy() if torch.is_tensor(audio) else audio)
        phonemes.append(ps)
    if verbose:
        print("Phonemes:", " | ".join(phonemes))
    return np.concatenate(chunks)

t0 = time.time()
y = kokoro_tts("Mary had a little lamb, its fleece was white as snow.", voice="bf_emma")
print(f"Generated {len(y)/SR:.1f}s of audio in {time.time()-t0:.2f}s")
add("kokoro_bf_emma_1", y, SR, MY_PROMPTS[1], "kokoro")

### Compare voices

Kokoro voice names encode accent and gender: `b`/`a` = British/American, `f`/`m` = female/male. We synthesise the same held-out sentences as you recorded, so we can compare them with your voice in Part 5.

In [ ]:
KOKORO_VOICES = ["bf_emma", "bf_isabella", "bm_george", "bm_lewis", "af_heart", "af_bella", "am_michael", "am_fenrir"]

for v in KOKORO_VOICES:
    for i in [1, 2, 3, 4]:
        y = kokoro_tts(MY_PROMPTS[i], voice=v, verbose=False)
        add(f"kokoro_{v}_{i}", y, SR, MY_PROMPTS[i], "kokoro", show=(i == 2))

### Predictions

TTS errors often come from the text front end (normalisation and G2P), not the neural model. Predict how each sentence will be read **before** running the cell, then listen and look at the phonemes.

* Heteronyms: *read* (present/past), *lead* (metal/verb), *live*, *wind*
* Numbers, dates, money, abbreviations
* Irish place names and names: are they in the pronunciation dictionary?

Kokoro lets you override a pronunciation with inline markdown-style phonemes: `[Kokoro](/kˈOkəɹO/)`.

In [ ]:
tests = [
    "I read the book yesterday, and I will read it again tomorrow.",
    "The lead singer was poisoned by lead pipes.",
    "On 3/4/2026 Dr. Smith paid €1,250.50 for 2 tickets at 7:30pm.",
    "Dún Laoghaire, Siobhán, Caoimhe and Tadhg went to Grangegorman.",
    "[Siobhán](/ʃəvˈɔn/) went to [Dún Laoghaire](/dʌn lˈiɹi/).",
]
for t in tests:
    print("\n" + t)
    display(Audio(kokoro_tts(t, voice="bm_george"), rate=SR))

In [ ]:
# Speaking rate is a simple global control
for s in [0.7, 1.0, 1.4]:
    print("speed", s); display(Audio(kokoro_tts(MY_PROMPTS[4], voice="bf_isabella", speed=s, verbose=False), rate=SR))

---
## Part 3: Zero-shot voice cloning with Qwen3-TTS

[Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS) (Alibaba Qwen, January 2026, Apache 2.0) is a **language-model-based TTS**. A neural audio codec turns speech into discrete tokens at 12.5 frames per second (16 codebooks per frame), and a transformer predicts those tokens from text, exactly as an LLM predicts word tokens. The `Base` models can clone a voice from about **3 seconds** of reference audio, with no training.

It offers two ways to condition on your voice:

| Mode | What the model sees | Analogy |
|---|---|---|
| `x_vector_only_mode=True` | Only a single **speaker embedding** vector computed from your clip | YourTTS / XTTS from previous labs |
| default (ICL, *in-context learning*) | The speaker embedding **plus** the codec tokens of your clip and its transcript, as a "prompt" to continue | Few-shot prompting an LLM |

We use the 0.6B model so it fits comfortably on a free T4 GPU. The first load downloads ~2.5 GB.

In [ ]:
from qwen_tts import Qwen3TTSModel

t0 = time.time()
qwen = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map=DEVICE, dtype=DTYPE, attn_implementation="sdpa",
)
print(f"Loaded in {time.time()-t0:.0f}s. Languages:", qwen.get_supported_languages())

### Speaker embedding only vs. in-context prompt

Predict: which of the two modes will sound **more like you**? Which will have the more natural **rhythm and intonation**? Why?

In [ ]:
def clone(text, ref_path=REF_PATH, ref_text=REF_TEXT, xvec=False, language="English", **gen):
    t0 = time.time()
    wavs, sr = qwen.generate_voice_clone(
        text=text, language=language, ref_audio=ref_path,
        ref_text=None if xvec else ref_text, x_vector_only_mode=xvec, **gen)
    y = wavs[0]
    print(f"  {len(y)/sr:.1f}s audio in {time.time()-t0:.1f}s")
    return y, sr

for i in [1, 2, 3, 4]:
    print(f"\n--- {MY_PROMPTS[i]}")
    y, sr = clone(MY_PROMPTS[i], xvec=True);  add(f"clone_xvec_{i}", y, sr, MY_PROMPTS[i], "clone_xvec", show=(i == 2))
    y, sr = clone(MY_PROMPTS[i], xvec=False); add(f"clone_icl_{i}", y, sr, MY_PROMPTS[i], "clone_icl", show=(i == 2))
print("\nYour real recording of the same sentence:"); display(Audio(BANK["me_2"]["y"], rate=SR))

###  how much reference audio is needed?

Cut your reference clip down to the first 3 s and 5 s (and give the model the matching *partial* transcript, roughly). Predict at what length the clone stops sounding like you.

In [ ]:
y_ref = BANK["me_0"]["y"]
for secs in [3, 5]:
    short = f"/content/my_voice/ref_{secs}s.wav"
    sf.write(short, y_ref[: int(secs * SR)], SR)
    print(f"\nReference = first {secs}s (x-vector mode, no transcript needed)")
    y, sr = clone(MY_PROMPTS[3], ref_path=short, xvec=True)
    add(f"clone_xvec{secs}s_3", y, sr, MY_PROMPTS[3], f"clone_xvec_{secs}s")

### Cross-lingual cloning and sampling

The same speaker identity can be carried into another language. Does your "accent" carry over too? Generation is **sampled** (like an LLM), so running twice gives different prosody; you can pass Hugging Face generation arguments such as `temperature` or `top_p`.

In [ ]:
y, sr = clone("Bonjour ! Je suis étudiant à Dublin et j'apprends les technologies du langage.", language="French"); display(Audio(y, rate=sr))
y, sr = clone("Hallo! Ich studiere in Dublin und lerne Sprachtechnologie.", language="German");                 display(Audio(y, rate=sr))
y, sr = clone("¡Hola! Estudio en Dublín y aprendo tecnologías del lenguaje.", language="Spanish");             display(Audio(y, rate=sr))

for temp in [0.3, 0.9, 1.3]:
    print("temperature", temp)
    y, sr = clone("Honestly, I did not expect this to work so well.", temperature=temp, do_sample=True); display(Audio(y, rate=sr))

### clone a Kokoro voice

A synthetic voice can be the reference too. Clone `bm_george` from one of his Kokoro clips. We will check in Part 5 whether the speaker-verification model thinks the clone is "George".

In [ ]:
george_ref = "/content/outputs/kokoro_bm_george_4.wav"
for i in [1, 2, 3]:
    y, sr = clone(MY_PROMPTS[i], ref_path=george_ref, ref_text=MY_PROMPTS[4])
    add(f"clone_of_george_{i}", y, sr, MY_PROMPTS[i], "clone_of_george", show=(i == 1))

---
## Part 4: Intelligibility via an ASR round trip

A standard automatic metric for TTS is to transcribe the synthetic speech with a strong ASR model and compute the **word error rate (WER)** against the input text. (This is how the Qwen3-TTS and most 2025–26 TTS papers report intelligibility; you used Whisper in the ASR lab.)



In [ ]:
from transformers import pipeline
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import jiwer, pandas as pd

asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32, device=DEVICE)
norm = BasicTextNormalizer()

def transcribe(y, sr):
    return asr({"raw": to16k(y, sr), "sampling_rate": 16000}, generate_kwargs={"language": "english"})["text"].strip()

rows = []
for name, d in BANK.items():
    if name == "me_0":            # reference clip, not a test sentence
        continue
    hyp = transcribe(d["y"], d["sr"])
    rows.append(dict(name=name, kind=d["kind"], wer=jiwer.wer(norm(d["text"]), norm(hyp)), hyp=hyp))
asr_df = pd.DataFrame(rows)
display(asr_df.groupby("kind")["wer"].agg(["mean", "count"]).sort_values("mean"))
pd.set_option("display.max_colwidth", 120)
display(asr_df[asr_df.wer > 0].sort_values("wer", ascending=False).head(10))

**Discussion:** Is WER a good measure of TTS *quality*? What does it miss (naturalness, prosody, speaker similarity)? What happens to WER when the ASR model and the TTS model were trained on similar data?

---
## Part 5: Speaker verification and identification

A **speaker embedding** model maps any utterance to a fixed-length vector so that utterances by the same speaker are close together, regardless of what was said. We use **ECAPA-TDNN** trained on VoxCeleb (≈7,000 celebrities), via SpeechBrain. It outputs a 192-dimensional embedding; we compare embeddings with **cosine similarity**.

Two tasks:
* **Verification** (1:1): *is this utterance by the claimed speaker?* Compare a test utterance with an enrolled voiceprint; accept if similarity > threshold. This is how "my voice is my password" phone banking works.
* **Identification** (1:N): *which of N enrolled speakers is this?* Pick the nearest voiceprint.

Note that Qwen3-TTS itself contains a speaker encoder (that is what `x_vector_only_mode` uses), so here we use a **different, independent** model as the judge.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

ecapa = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                       savedir="/content/models/ecapa", run_opts={"device": DEVICE})

@torch.no_grad()
def embed(y, sr):
    wav = torch.tensor(to16k(y, sr), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    e = ecapa.encode_batch(wav).squeeze()
    return F.normalize(e, dim=0).cpu().numpy()

EMB = {name: embed(d["y"], d["sr"]) for name, d in BANK.items()}
print(len(EMB), "embeddings of dim", next(iter(EMB.values())).shape)

### Enrol yourself

Your **voiceprint** is the mean embedding of your reference clip plus one held-out sentence. The other three real recordings are genuine test trials.

In [ ]:
def voiceprint(names):
    v = np.mean([EMB[n] for n in names], axis=0)
    return v / np.linalg.norm(v)

ENROL = ["me_0", "me_1"]
my_vp = voiceprint(ENROL)
scores = {n: float(EMB[n] @ my_vp) for n in BANK if n not in ENROL}

kinds = {}
for n, s in scores.items():
    kinds.setdefault(BANK[n]["kind"], []).append(s)

plt.figure(figsize=(10, 4))
for k, v in sorted(kinds.items()):
    plt.scatter(v, [k] * len(v), s=60, alpha=0.7)
THRESH = 0.25    # SpeechBrain's default threshold for this model
plt.axvline(THRESH, color="r", ls="--", label=f"threshold = {THRESH}")
plt.xlabel("cosine similarity to YOUR voiceprint"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for k, v in sorted(kinds.items(), key=lambda kv: -np.mean(kv[1])):
    acc = np.mean(np.array(v) > THRESH)
    print(f"{k:18s} mean sim = {np.mean(v):.3f}   accepted as you: {acc:.0%}")

### Questions

* Which kind of clone gets closest to your real recordings? Is that the same one that *sounded* most like you?
* Where would you set the threshold so that **all** your real recordings are accepted? Do any clones get accepted at that threshold? This is the trade-off between the **false rejection rate** and the **false acceptance rate**; the point where they are equal is the **equal error rate (EER)**.
* A bank using voice authentication would want a very low false acceptance rate. What does this experiment suggest about voice biometrics in 2026?